In [ ]:
import math
from functools import partial

import jax
import numpy as np
import jax.numpy as jnp
from jax.sharding import Mesh

import tiktoken

from model import GPT, forward
from model import precompute_frequencies
from model import forward_infer
from kvcache import KVCache, count_left_padding, prepare_chunk
from kvcache import segment_ids_to_positions
from checkpoint_utils import load_weights_from_checkpoint_with_validation
from config import ShardingRules, Config, BATCH_AXIS_NAME
from quantization import quantize_params, QArray

In [2]:
def build_tokenizer():
    """Build a GPT-2 tokenizer extended with custom chat tokens."""
    pad_token = "<|pad|>"

    custom_tokens = [pad_token,]
    base = tiktoken.get_encoding("gpt2")
    custom_token_ids = {tok: base.n_vocab + i for i, tok in enumerate(custom_tokens)}

    tokenizer = tiktoken.Encoding(
        name="gpt2_with_custom_tokens",
        pat_str=base._pat_str,
        mergeable_ranks=base._mergeable_ranks,
        special_tokens={**base._special_tokens, **custom_token_ids},
    )

    bos_id = tokenizer.eot_token
    bos = tokenizer.decode([bos_id])

    return {
        "tokenizer": tokenizer,
        "bos_id": bos_id,
        "bos": bos,
        "pad_token": pad_token,
        "pad_id": custom_token_ids[pad_token],
        "custom_token_ids": custom_token_ids,
        "vocab_size": tokenizer.n_vocab,
    }

def pad_tokens(tokens, pad_id, pad_to_power_of_two=False):
    curr_max_len = max([len(s) for s in tokens])
    if pad_to_power_of_two:
        pad_to = 2 ** math.ceil(math.log2(curr_max_len))
    else:
        pad_to = curr_max_len
    padded = []
    segment_ids = []

    for encoded in tokens:
        p, s = prepare_chunk(jnp.array(encoded), pad_id=pad_id, pad_to=pad_to)
        padded.append(p[0])
        segment_ids.append(s[0])

    padded = jnp.stack(padded)
    segment_ids = jnp.stack(segment_ids)
    return padded, segment_ids

In [ ]:
devices = np.array(jax.devices())
mesh = Mesh(devices, axis_names=BATCH_AXIS_NAME)
sharding_rules = ShardingRules(batch=BATCH_AXIS_NAME)
cfg = Config(mesh=mesh, rules=sharding_rules)

# Load model weights
model = GPT.init(jax.random.PRNGKey(0), cfg)
model_sharding = GPT.shardings(cfg.mesh, cfg.rules, cfg.model)
model = load_weights_from_checkpoint_with_validation(cfg.ckpt_cfg.load_params_ckpt_path, model, model_sharding)

# Qunatize model params
model, stats = quantize_params(model, include_embeddings=True, return_stats=True)

# Build the tokenizer
tok_info = build_tokenizer()
tokenizer = tok_info["tokenizer"]
PAD_ID = tok_info["pad_id"]
BOS_ID = tok_info["bos_id"]
PAD_TOKEN = tok_info["pad_token"]
BOS_TOKEN = tok_info["bos"]
head_dim = cfg.model.attn.head_dim

In [4]:
@partial(jax.jit, static_argnames=("head_dim", "pad_id"))
def prefill(params, input_ids, segment_ids, cache, head_dim, pad_id):
    """Prefill with LEFT-padded sequences for the absolute-position cache."""
    left_pad_counts = count_left_padding(input_ids, pad_id=pad_id)
    cache = cache.with_left_padding(left_pad_counts)
    logits, cache = forward_infer(params, input_ids, segment_ids, cache, head_dim)
    last_token_logits = logits[:, -1, :]
    next_tokens = jnp.argmax(last_token_logits, axis=-1)
    return logits, next_tokens, cache


def decode(params, input_ids, cache, head_dim):
    """Decode step for LEFT-padded sequences aligned at the right edge."""
    segment_ids = jnp.ones_like(input_ids, dtype=jnp.int32)
    logits, cache = forward_infer(params, input_ids, segment_ids, cache, head_dim)
    return logits[:, -1, :], cache


def sample_from_logits(logits, rng, temperature=1.0, top_k=0):
    vocab = logits.shape[-1]

    if temperature <= 0.0:
        return jnp.argmax(logits, axis=-1).astype(jnp.int32)
    else:
        tiny = jnp.finfo(jnp.float32).tiny
        logits = logits / jnp.maximum(temperature, tiny)

    if top_k is not None:
        k = int(top_k)
        if 0 < k < vocab:

            def set_values(arr, idx, val):
                return arr.at[idx].set(val)

            values, indices = jax.lax.top_k(logits, k)
            filtered_logits = jnp.full_like(logits, fill_value=-jnp.inf)
            logits = jax.vmap(set_values, in_axes=(0, 0, 0))(
                filtered_logits, indices, values
            )
    return jax.random.categorical(rng, logits, axis=-1).astype(jnp.int32)


@partial(jax.jit, static_argnames=("temperature", "head_dim", "max_new_tokens", "top_k"))  # fmt: off
def generate(
    params,
    cache,
    last_token,
    generated_tokens,
    head_dim,
    decode_key,
    temperature,
    top_k,
    max_new_tokens,
):
    def decode_body(carry, t):
        cache, last_token, decode_key, generated_tokens = carry
        logits, cache = decode(params, last_token, cache, head_dim)
        decode_key, sub = jax.random.split(decode_key)
        token = sample_from_logits(logits, sub, temperature, top_k)
        generated_tokens = generated_tokens.at[:, t].set(token)
        return (cache, token[:, None], decode_key, generated_tokens), None

    (cache, last_token, decode_key, generated_tokens), _ = jax.lax.scan(
        decode_body,
        (cache, last_token, decode_key, generated_tokens),
        jnp.arange(1, max_new_tokens),
    )
    return generated_tokens


@partial(jax.jit, static_argnames=("head_dim", "max_new_tokens", "temperature", "top_k", "stop_token_id"))  # fmt: off
def generate_with_stop_condition(
    params,
    cache,
    last_token,
    generated_tokens,
    head_dim,
    decode_key,
    temperature,
    top_k,
    max_new_tokens,
    stop_token_id,
):
    def cond_fn(state):
        _, _, _, _, step, finished = state
        return (~finished) & (step < max_new_tokens)

    def body_fn(state):
        cache, last_token, decode_key, generated_tokens, step, finished = state
        logits, cache = decode(params, last_token, cache, head_dim)
        decode_key, sub = jax.random.split(decode_key)
        token = sample_from_logits(logits, sub, temperature, top_k)
        generated_tokens = generated_tokens.at[:, step].set(token)
        finished = jnp.all(token == stop_token_id)
        return (cache, token[:, None], decode_key, generated_tokens, step + 1, finished)

    state = (
        cache,
        last_token,
        decode_key,
        generated_tokens,
        jnp.int32(1),
        jnp.bool_(False),
    )
    _, _, _, generated_tokens, _, _ = jax.lax.while_loop(cond_fn, body_fn, state)
    return generated_tokens

In [5]:
prompts = [
        "<|endoftext|>Did you notice that this world",
        "<|endoftext|>Hello World! My dear",
        "<|endoftext|>Some say we are tired far",
        "<|endoftext|>Hear that?",
    ]

encoded = tokenizer.encode_batch(prompts, allowed_special="all")
input_ids, segment_ids = pad_tokens(encoded, pad_to_power_of_two=True, pad_id=PAD_ID)
cache_key = jax.random.PRNGKey(1)
batch_size = input_ids.shape[0]
cache = KVCache.init(cache_key, cfg.mesh, cfg.rules, batch_size, cfg)
temperature = 0.8
top_k = 100
max_new_tokens = 100

In [6]:
with jax.set_mesh(cfg.mesh):
    logits, next_tokens, cache = prefill(model, input_ids, segment_ids, cache, head_dim, pad_id=PAD_ID)
print(next_tokens, tokenizer.decode(next_tokens), prefill._cache_size())

[ 318 1545 1165  632]  is friend too It 1


In [7]:
with jax.set_mesh(cfg.mesh):
    generated_tokens = (jnp.zeros((batch_size, max_new_tokens), dtype=jnp.int32).at[:, 0].set(next_tokens))
    last_token = next_tokens[:, None]
    generated = generate(model, cache, last_token, generated_tokens, head_dim, jax.random.PRNGKey(5) , temperature=temperature, top_k=top_k, max_new_tokens=max_new_tokens)

generate._cache_size()

1

In [8]:
decoded = tokenizer.decode_batch(generated.tolist())
for p, d in zip(prompts, decoded):
    print(p+d)
    print("-"*75)

<|endoftext|>Did you notice that this world is filled with so many people and animals that have lost their lives?
Every world has a story, that is a story worth sharing and it is important to keep in mind.
It is not always easy to find someone who is not dead at all. You need to be brave enough to find a way to bring someone home.
You also need to be aware of the challenges we face. If you are in a situation where you are not sure of your options, come to me at the
---------------------------------------------------------------------------
<|endoftext|>Hello World! My dear friend,
This is my first post in our new adventure with You!
I’m still going strong, because we have seen a lot of the changes to the world, but the last few weeks have brought about more drastic changes.
Today I’m going to talk about everything about the new adventure, to help you in the right direction.
The new adventure was called The Legend of Zelda: The Four Kingdoms for the 3DS (which I am quite happy about, as